# script 3: carga a la nube (minio-to-azure)
aqui copiamos los datos parquet desde el minion local hacia la cuenta de almacenamiento en azure que nos dio el profe (adls gen2).

> **carpeta en azure:** `abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_7`
> **seguridad:** cero credenciales quemadas en el codigo. todo se lee dinamicamente desde `.dlt/secrets.toml`.

In [ ]:
import dlt
from dlt.sources.filesystem import readers, read_parquet

# 1. configuramos el pipeline de dlt hacia azure
pipeline = dlt.pipeline(
    pipeline_name="s3_to_adls",
    destination="filesystem",
    dataset_name="taxis_parquet",
)

# 2. lector para los archivos parquet que estan en el minion
parquet_reader = readers(
    bucket_url="s3://taxis/taxis_parquet/df_data/",
    file_glob="*.parquet"
).read_parquet()

# le ponemos el nombre a la tabla
parquet_reader = parquet_reader.with_name("df_parquet")

# 3. mandamos los datos a la nube de azure del profe
print("subiendo datos a azure data lake (GRUPO_7)...")
load_info = pipeline.run(
    parquet_reader,
    loader_file_format="parquet",
    write_disposition="replace"
)

print("=== reporte de carga a azure ===")
print(load_info)
print(pipeline.last_trace.last_normalize_info)

subiendo datos a azure data lake (GRUPO_7)...


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
2026-09-06 15:29:21,600|[INFO]|5061|127336686163776|dlt|pool_runner.py|create_pool:203|Created none pool with 1 workers
2026-09-06 15:29:21,974|[INFO]|5061|127336686163776|dlt|normalize.py|run:309|Running file normalizing
2026-09-06 15:29:21,983|[INFO]|5061|127336686163776|dlt|normalize.py|run:312|Found 1 load packages
2026-09-06 15:29:22,014|[INFO]|5061|127336686163776|dlt|normalize.py|run:335|Found 2 files in schema s3_to_adls load_id 1788708307.9683847
2026-09-06 15:29:22,289|[INFO]|5061|127336686163776|dlt|normalize.py|spool_sch

In [5]:
# 4. verificacion del archivo cargado en azure (investigacion requerida en la rubrica)
import adlfs
import dlt

# leemos las credenciales desde dlt.secrets sin quemar ninguna clave en el codigo
azure_creds = dlt.secrets.get("s3_to_adls.destination.filesystem.credentials")
acc_name = azure_creds.get("azure_storage_account_name")
acc_key = azure_creds.get("azure_storage_account_key")

# nos conectamos al sistema de archivos de azure
fs_azure = adlfs.AzureBlobFileSystem(
    account_name=acc_name,
    account_key=acc_key
)

# listamos los archivos que quedaron en la carpeta de nuestro grupo
archivos_subidos = fs_azure.glob("clase-4-dlt/GRUPO_7/**/*.parquet")
print("archivos encontrados en azure para el GRUPO_7:")
for f in archivos_subidos:
    detalles = fs_azure.info(f)
    tam_mb = detalles['size'] / (1024 * 1024)
    print(f" - {f} ({tam_mb:.2f} MB)")

archivos encontrados en azure para el GRUPO_7:
 - clase-4-dlt/GRUPO_7/taxis_parquet/df_parquet/1788708307.9683847.5d21deb955.parquet (138.83 MB)
